# Taylor Approximation of Price Changes

Duration and convexity (01/02) are the first two terms of a Taylor series: they replace the bond price curve near today's yield by a
parabola. How good is that parabola? The answer has a precise form: the error of a first-order approximation shrinks like the square of
the yield change, the error of a second-order approximation like its cube. This notebook derives the expansion, measures the errors for
small and large yield moves, and computes "effective" duration and convexity by bumping the yield, the method risk systems use for
bonds whose price has no simple formula, such as callable bonds with negative convexity.

## 0. Math link

- Taylor series: **gap**. LearningCalculus does not yet have a Taylor-series notebook; this notebook states the result it needs.
  The nearest prerequisites are
  [higher-order derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/09_higher_order_derivatives.ipynb)
  and the [geometric interpretation of the derivative](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/11_geometric_and_physical_interpretation_of_derivative.ipynb)
  (the tangent line is the first-order Taylor polynomial).
- Duration and convexity from [01/02](./02_duration_and_convexity.ipynb).

## 1. Introduction

A risk manager wants to know what happens to a bond book if yields jump by 1.5 to 2 percentage points, as ten-year Swiss yields did in 2022. Full
revaluation of every bond is exact but slow; duration alone is fast but misses the curvature. A second-order Taylor approximation,
duration plus convexity, is the standard compromise. Knowing its error lets you decide when it is good enough.

After this notebook you can:

- write the Taylor expansion of a bond price around the current yield, with duration, convexity and a third-order term;
- predict how the approximation error scales with the size of the yield change, and verify it numerically;
- compute effective duration and convexity by finite differences;
- explain negative convexity of callable bonds and why duration-only hedges fail for them.

## 2. Theory

### Taylor's theorem

If $f$ has $k + 1$ continuous derivatives near $x_0$, then for a change $h$

$$
f(x_0 + h) = f(x_0) + f'(x_0)\,h + \frac{f''(x_0)}{2!}h^2 + \dots + \frac{f^{(k)}(x_0)}{k!}h^k + R_k(h),
\qquad R_k(h) = \frac{f^{(k+1)}(\xi)}{(k+1)!}h^{k+1}
$$

for some $\xi$ between $x_0$ and $x_0 + h$ (Lagrange form of the remainder). The polynomial of degree $k$ matches the function's value
and first $k$ derivatives at $x_0$; the remainder is of order $h^{k+1}$.

### Applied to a bond

With $P(y) = \sum_t C_t (1+y)^{-t}$, every derivative exists for $y > -1$:

$$
P^{(k)}(y) = (-1)^k \sum_t t(t+1)\cdots(t+k-1)\, C_t (1+y)^{-t-k} .
$$

Dividing by $P$ and using $D_{\text{mod}} = -P'/P$ and $\text{Conv} = P''/P$:

$$
\frac{\Delta P}{P} = -D_{\text{mod}}\,\Delta y + \tfrac12\,\text{Conv}\,(\Delta y)^2 + \tfrac16\,\frac{P'''}{P}(\Delta y)^3 + O(\Delta y^4) .
$$

The third derivative is negative for bonds with positive cash flows, so the cubic term is negative for $\Delta y > 0$ and positive for
$\Delta y < 0$. The second-order estimate therefore understates the loss when yields rise and understates the gain when they fall.

### Size of the errors

| approximation | error | doubling $\Delta y$ multiplies the error by |
|---|---|---|
| first order (duration) | $\approx \tfrac12\text{Conv}\,(\Delta y)^2$ | 4 |
| second order (duration + convexity) | $\approx \tfrac16 \frac{P'''}{P}(\Delta y)^3$ | 8 |

For a 10-year bond with convexity around 100, the duration-only error for a one-point move is about $\tfrac12 \times 100 \times 0.01^2 = 0.5\%$
of the price; for a three-point move about 4.5%.

### Effective duration and convexity

When the price is computed by a model (a tree, a Monte Carlo simulation, a pricing library) rather than a formula, the derivatives are
estimated by bumping the yield up and down by $h$:

$$
D_{\text{eff}} = \frac{P(y - h) - P(y + h)}{2h\,P(y)}, \qquad \text{Conv}_{\text{eff}} = \frac{P(y + h) - 2P(y) + P(y - h)}{h^2\,P(y)} .
$$

These are central differences; by Taylor's theorem their error is of order $h^2$.

### Negative convexity

A callable bond can be redeemed by the issuer at a fixed call price $K$. When yields fall and the straight bond's price rises towards
$K$, the issuer is likely to call, so the callable bond's price is capped near $K$. Its price curve bends the other way near the cap: its
convexity is **negative**. Owners of callable bonds (and of mortgages that borrowers can prepay) lose more from rising yields than they
gain from falling ones.

## 3. Worked Examples

### Example 1: a 30-year zero at 2%, yields up 2 points

$P = 100/1.02^{30} = 55.21$. Modified duration $29.41$, convexity $893.9$ (01/02). First order: $-29.41 \times 0.02 = -58.8\%$. Second
order: $-58.8\% + \tfrac12 \times 893.9 \times 0.0004 = -40.9\%$. Exact: $1.02^{30}/1.04^{30} - 1 = -44.2\%$. Duration alone overstates
the loss by 14.7 points; adding convexity leaves an error of 3.2 points (the other way).

### Example 2: the same bond, yields up 0.25 points

First order $-7.35\%$, second order $-7.074\%$, exact $-7.081\%$. For small moves the second-order error is below a hundredth of a percent.

### Example 3: the third-order term

For a zero, $P'''/P = -n(n+1)(n+2)/(1+y)^3 = -28{,}043$ for $n = 30$, $y = 2\%$. The cubic term for $\Delta y = 2\%$ is
$\tfrac16 \times (-28{,}043) \times 0.02^3 = -3.7\%$, slightly more than the remaining 3.2-point gap: the third-order estimate,
$-44.7\%$, is within about half a point of the exact $-44.2\%$.

### Example 4: effective duration by bumping

Bump the 30-year zero's yield by $h = 0.01\%$: $P(1.99\%) = 55.3697$, $P(2.01\%) = 55.0450$. Then
$D_{\text{eff}} = (55.3697 - 55.0450)/(2 \times 0.0001 \times 55.2071) = 29.41$, the analytic modified duration.

## 4. Finance Application

### Where it matters

- **Stress tests.** Regulators (FINMA, the Basel IRRBB standard) require the effect of parallel shocks of ±1 to ±3 points; second-order
  approximations are a quick check against full revaluation.
- **Risk systems.** Sensitivities ("Greeks") of every instrument are computed by bumping inputs; the P&L of a book is then explained by a
  Taylor expansion in those sensitivities (P&L attribution).
- **Callable bonds and mortgages.** Negative convexity makes hedges based on duration alone fail when rates move a lot; mortgage investors
  buy options to restore convexity.
- **Options.** The same expansion in the underlying price gives delta and gamma (module 10).

### The usual misreadings

- "Duration tells me the loss for any rate move." The error grows with the square of the move; for 2 points on a long bond it is huge.
- "Adding convexity makes the estimate exact." The error still grows with the cube of the move.
- "Convexity is always good." Positive convexity is; callable bonds and mortgages have negative convexity.
- "A smaller bump is always more accurate." Too small a bump amplifies rounding error in the model prices; $h$ between 1 and 10 basis
  points is typical.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### Derivatives of any order, and the Taylor estimate

In [ ]:
def bond_cash_flows(coupon, years, face=100.0):
    cf = np.full(years, coupon * face)
    cf[-1] += face
    return cf

def price(cf, y):
    t = np.arange(1, len(cf) + 1)
    return np.sum(cf / (1 + y) ** t)

def derivative(cf, y, k):
    # k-th derivative of the price with respect to y
    t = np.arange(1, len(cf) + 1)
    rising = np.ones_like(t, dtype=float)
    for j in range(k):
        rising *= t + j                    # t (t+1) ... (t+k-1)
    return (-1) ** k * np.sum(rising * cf / (1 + y) ** (t + k))

def taylor_change(cf, y, dy, order):
    p = price(cf, y)
    total, fact = 0.0, 1.0
    for k in range(1, order + 1):
        fact *= k
        total += derivative(cf, y, k) / fact * dy**k
    return total / p                       # relative price change

zero30 = bond_cash_flows(0.0, 30)
y0 = 0.02
for dy in [0.0025, 0.02]:
    exact = price(zero30, y0 + dy) / price(zero30, y0) - 1
    approx = {k: taylor_change(zero30, y0, dy, k) for k in [1, 2, 3]}
    print(f"dy = {dy:.2%}: exact {exact:.4%}, first {approx[1]:.4%}, second {approx[2]:.4%}, third {approx[3]:.4%}")
print("P'''/P =", round(derivative(zero30, y0, 3) / price(zero30, y0), 1))

For a quarter-point move all three approximations are within 0.3 points; the second-order one is within 0.03. For a two-point move duration
alone is off by almost 15 points; duration plus convexity by about 3 points; the third order brings it within half a point.

### Error against the size of the move

In [ ]:
cf10 = bond_cash_flows(0.015, 10)
rows = []
for dy in [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03]:
    for name, cf in [("10y 1.5%", cf10), ("30y zero", zero30)]:
        exact = price(cf, y0 + dy) / price(cf, y0) - 1
        rows.append({"bond": name, "dy (%)": dy * 100,
                     "exact (%)": exact * 100,
                     "error, 1st order (pp)": (taylor_change(cf, y0, dy, 1) - exact) * 100,
                     "error, 2nd order (pp)": (taylor_change(cf, y0, dy, 2) - exact) * 100})
errors = pd.DataFrame(rows)
errors.pivot(index="dy (%)", columns="bond", values=["error, 1st order (pp)", "error, 2nd order (pp)"])

Doubling the move from 0.5 to 1 point multiplies the first-order error by about 4 and the second-order error by about 8, as the
remainder term predicts. For the ten-year bond, the second-order error stays below 0.02 points up to a one-point move and is about 0.5 points
at three points; for the 30-year zero it reaches 3 points at a two-point move and 10 points at three.

### Effective duration and convexity, and the choice of bump

In [ ]:
p0 = price(zero30, y0)
true_D = -derivative(zero30, y0, 1) / p0
true_C = derivative(zero30, y0, 2) / p0
rows = []
for h in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]:
    p_up, p_dn = price(zero30, y0 + h), price(zero30, y0 - h)
    rows.append({"bump h": h,
                 "duration error": (p_dn - p_up) / (2 * h * p0) - true_D,
                 "convexity error": (p_up - 2 * p0 + p_dn) / (h**2 * p0) - true_C})
pd.DataFrame(rows).set_index("bump h").style.format("{:.2e}")

The errors first fall like $h^2$ (truncation error from the Taylor remainder), then rise again for tiny bumps, where the difference of two
nearly equal prices loses digits to floating-point rounding. Convexity, which divides by $h^2$, suffers earlier. With these exact
formula prices the best bump is about 0.1 basis point ($10^{-5}$); prices from a tree or a simulation carry their own noise, and 1 to 10
basis points are typical in practice.

### A stylized callable bond

A ten-year 3% bond callable at 102. We cap the straight price smoothly at the call price, $P_{\text{call}} = -s\ln\big(e^{-P/s} + e^{-K/s}\big)$
with $s = 1$: a "soft minimum" of the straight price and the call price. Real callable bonds are priced with interest-rate trees; the
shape is the same.

In [ ]:
cf_call = bond_cash_flows(0.03, 10)
K, s = 102.0, 1.0

def callable_price(y):
    p = price(cf_call, y)
    return -s * np.logaddexp(-p / s, -K / s)

def effective(f, y, h=1e-4):
    p0_, p_up, p_dn = f(y), f(y + h), f(y - h)
    return (p_dn - p_up) / (2 * h * p0_), (p_up - 2 * p0_ + p_dn) / (h**2 * p0_)

rows = []
for y in [0.01, 0.02, 0.025, 0.03, 0.04, 0.05]:
    d_s, c_s = effective(lambda x: price(cf_call, x), y)
    d_c, c_c = effective(callable_price, y)
    rows.append({"yield": y, "straight price": price(cf_call, y), "callable price": callable_price(y),
                 "straight duration": d_s, "callable duration": d_c,
                 "straight convexity": c_s, "callable convexity": c_c})
pd.DataFrame(rows).set_index("yield")

At high yields the call is irrelevant and both bonds have the same duration and convexity. As yields fall towards the level where the
straight price exceeds 102, the callable bond's duration shrinks and its convexity turns strongly negative: its price stops rising.

## 6. Visualizations

In [ ]:
dys = np.linspace(-0.03, 0.03, 200)
exact = np.array([price(zero30, y0 + d) / p0 - 1 for d in dys])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dys * 100, exact * 100, "k", lw=2.5, label="exact")
for k, style in [(1, "--"), (2, "-."), (3, ":")]:
    ax.plot(dys * 100, [taylor_change(zero30, y0, d, k) * 100 for d in dys], style, label=f"Taylor, order {k}")
ax.set_title("30-year zero at 2%: price change and its Taylor approximations")
ax.set_xlabel("yield change (percentage points)")
ax.set_ylabel("price change (%)")
ax.set_ylim(-100, 160)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The tangent line (order 1) is good only near zero. The parabola (order 2) follows the curve over a wider range but lies above it for
large rises (too small a loss) and below it for large falls (too small a gain); the cubic term, negative for rises and positive for falls,
corrects both. Each extra order widens the range where the approximation is usable.

In [ ]:
small = np.logspace(-4, np.log10(0.03), 40)
fig, ax = plt.subplots(figsize=(9, 5))
for k in [1, 2, 3]:
    err = [abs(taylor_change(zero30, y0, d, k) - (price(zero30, y0 + d) / p0 - 1)) for d in small]
    ax.loglog(small * 100, np.array(err) * 100, label=f"order {k}")
ref = small * 100
ax.loglog(ref, 0.5 * true_C * small**2 * 100, "k:", lw=1, label="slopes 2, 3, 4 (reference)")
ax.loglog(ref, abs(derivative(zero30, y0, 3) / p0) / 6 * small**3 * 100, "k:", lw=1)
ax.loglog(ref, abs(derivative(zero30, y0, 4) / p0) / 24 * small**4 * 100, "k:", lw=1)
ax.set_title("Approximation error against the size of the yield change, 30-year zero")
ax.set_xlabel("yield change (percentage points, log scale)")
ax.set_ylabel("absolute error (percentage points, log scale)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

On log–log axes the errors are straight lines with slopes 2, 3 and 4: the order of the first neglected term. Each order buys one more
power of $\Delta y$.

In [ ]:
ys = np.linspace(0.0, 0.06, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(ys * 100, [price(cf_call, y) for y in ys], label="straight 3% bond")
axes[0].plot(ys * 100, [callable_price(y) for y in ys], label="callable at 102")
axes[0].axhline(K, color="grey", ls=":", lw=1)
axes[0].set_title("Price against yield")
axes[0].set_xlabel("yield (% per year)")
axes[0].set_ylabel("price (CHF per 100 face)")
axes[0].legend()

axes[1].plot(ys * 100, [effective(lambda x: price(cf_call, x), y)[1] for y in ys], label="straight")
axes[1].plot(ys * 100, [effective(callable_price, y)[1] for y in ys], label="callable")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Effective convexity")
axes[1].set_xlabel("yield (% per year)")
axes[1].set_ylabel("convexity")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The callable bond's price flattens under the call price, and its convexity becomes negative in the region around the yield where the
straight bond would be worth 102, about 2.8%. There the bond behaves like a short option position: it gains little when yields fall and
loses the full duration when they rise.

## 7. Exercises

### Basic

1. A bond has a modified duration of 8 and a convexity of 80. Estimate the price change for yield changes of +1 and −1 point with the
   first- and second-order approximations.
2. Write down the second-order Taylor polynomial of $\ln(1 + r)$ around $r = 0$ and use it to explain the approximation $\ell \approx r - r^2/2$
   from 00/01.
3. A risk report shows a duration-only estimate of a −6% loss for a +1.5 point shock on a long bond portfolio, and the full revaluation
   shows −5.2%. **Interpret** the difference. Which number would you report to the board, and why?

### Intermediate

4. For the ten-year 1.5% bond, find the yield change at which the first-order error reaches 1% of the price. Compare with the prediction
   $\sqrt{2 \times 0.01 / \text{Conv}}$.
5. Compute effective duration and convexity of the callable bond at 3% with bumps of 1, 10 and 50 basis points. Why do the large bumps give
   different answers?

### Advanced

6. Show that for a zero-coupon bond the full Taylor series in $\Delta y$ is the binomial series of $(1 + \Delta y/(1+y))^{-n}$, and find its
   radius of convergence. What happens for $\Delta y < -(1+y)$?
7. A portfolio is long a 30-year zero and short a 10-year zero with the same DV01. Use the second-order approximation to sketch its P&L
   against a parallel shift, and verify with full revaluation. What sign does its convexity have?

## 8. Solutions / Checks

In [ ]:
# 1.
for dy in [0.01, -0.01]:
    print(f"1. dy = {dy:+.0%}: first order {-8 * dy:.2%}, second order {-8 * dy + 0.5 * 80 * dy**2:.2%}")

# 4.
p10 = price(cf10, y0)
C10 = derivative(cf10, y0, 2) / p10
dys = np.linspace(0.0001, 0.05, 5000)
err1 = np.array([abs(taylor_change(cf10, y0, d, 1) - (price(cf10, y0 + d) / p10 - 1)) for d in dys])
print(f"4. first-order error reaches 1% at dy = {dys[np.argmax(err1 >= 0.01)]:.2%}; "
      f"prediction sqrt(2 x 0.01 / Conv) = {np.sqrt(2 * 0.01 / C10):.2%}")

# 5.
for h in [1e-4, 1e-3, 5e-3]:
    d_c, c_c = effective(callable_price, 0.03, h)
    print(f"5. bump {h * 1e4:4.0f} bp: duration {d_c:.3f}, convexity {c_c:.1f}")

# 7.
p30, p10z = price(zero30, y0), price(bond_cash_flows(0.0, 10), y0)
dv30, dv10 = -derivative(zero30, y0, 1), -derivative(bond_cash_flows(0.0, 10), y0, 1)
n10 = dv30 / dv10                                       # units of the 10-year zero to short
for dy in [-0.02, -0.01, 0.01, 0.02]:
    pnl = (price(zero30, y0 + dy) - p30) - n10 * (price(bond_cash_flows(0.0, 10), y0 + dy) - p10z)
    approx = 0.5 * (derivative(zero30, y0, 2) - n10 * derivative(bond_cash_flows(0.0, 10), y0, 2)) * dy**2
    print(f"7. dy {dy:+.0%}: P&L {pnl:.3f}, second-order estimate {approx:.3f} (CHF per 100 face of the 30-year zero)")

**2.** $\ln(1 + r) = r - r^2/2 + O(r^3)$: the derivatives at 0 are $1$ and $-1$, so the quadratic term is $-r^2/2$.

**3.** The 0.8-point gap is the convexity gain: the price curve lies above its tangent, so the true loss is smaller. Report the full
revaluation (−5.2%), with the duration estimate as a cross-check. If the portfolio contained callable bonds or mortgages, the
true loss could also be larger than the duration estimate.

**5.** For the callable bond the price curve bends sharply near the call region; a large bump averages the slope over a wide range of
yields and smooths the convexity away. The 1 bp and 10 bp results are close; the 50 bp bump is not local any more.

**6.** $P = F(1+y)^{-n}(1 + x)^{-n}$ with $x = \Delta y/(1+y)$, and $(1 + x)^{-n} = \sum_k \binom{-n}{k}x^k$ converges for $|x| < 1$, i.e.
$|\Delta y| < 1 + y$. For $\Delta y \le -(1+y)$ the new yield is $\le -100\%$ and the price is undefined: no approximation can work there.

**7.** With equal DV01 the first-order term vanishes; the 30-year zero has much more convexity per unit of DV01, so the P&L is positive for
moves in either direction and the position has positive convexity. In practice this convexity is paid for by a lower yield on the long bond.

## 9. Conclusion

- A bond price has a Taylor series in the yield change; duration and convexity are its first two scaled coefficients.
- First-order (duration) errors grow like $(\Delta y)^2$, second-order (with convexity) like $(\Delta y)^3$; each extra order adds a power.
- For long bonds and large shocks the second-order estimate can still be off by several points; full revaluation is the reference.
- Effective duration and convexity come from bumping the yield; the bump size trades truncation error ($h^2$) against rounding error.
- Callable bonds and mortgages have negative convexity near the call region, so duration-only hedges fail exactly when rates move most.

Next: [01/04 Yield curves and forward rates](./04_yield_curves_and_forward_rates.ipynb).